# 02 · Geometry QA — double counting, parse errors, network islands

Sources: OpenStreetMap India via Geofabrik (ODbL) · MoRTH Annual Report 2024-25, Appendix-2 (as on 31.12.2024) · WorldPop 2020 1 km (CC BY 4.0) · GHSL built-up (CC BY 4.0) · MoRTH Road Accidents in India 2024 · NHAI Datalake — aggregates only (ADR-0014). Rebuild everything: `cd ml && uv run python -m fields.national_highways all`.

In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ML = Path.cwd().resolve().parents[1]  # ml/notebooks/national_highways -> ml
sys.path.insert(0, str(ML))
from fields.national_highways import analysis as nh

A = nh.A
SUMMARY = json.loads((A / "summary.json").read_text())
pd.set_option("display.max_rows", 40)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams.update({"figure.dpi": 80, "figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})

In [2]:
seg = nh.segments()
op = seg[seg.status == "operational"]
print(f"Raw length {op.length_km.sum():,.0f} km -> one carriageway {op.eff_km.sum():,.0f} km; "
      f"one-way ways {op.oneway.mean():.0%}, paired {(op[op.oneway].weight == 0.5).mean():.0%}")
op.groupby("road_class")[["length_km", "eff_km"]].sum().round(0)

Raw length 199,362 km -> one carriageway 146,322 km; one-way ways 75%, paired 94%


,length_km,eff_km
road_class,,
motorway,"14,457.00","8,774.00"
primary,"2,800.00","2,283.00"
secondary,676.00,626.00
tertiary,"1,212.00","1,043.00"
trunk,"180,217.00","133,596.00"


In [3]:
# where the NH ref came from: the way itself, its NH route relation, or none (trunk without a ref)
op.groupby(op.ref_source.fillna("none")).eff_km.sum().round(0)

ref_source
none         5,823.00
relation     2,768.00
way        137,731.00
Name: eff_km, dtype: float64

## Ingest errors (fail closed: logged, never guessed)

In [4]:
err = pd.read_csv(nh.OUT / "ingest_errors.csv")
display(err.reason.value_counts())
err[err.reason == "unparsed_ref"].value.value_counts().head(10)

reason
unparsed_ref    19
no_state         3
Name: count, dtype: int64

value
NH52<unterschiedlich>        7
NH 31 (C)                    6
NH                           3
NH168#                       1
NH968.                       1
NH16 LANJIPALLI BERHAMPUR    1
Name: count, dtype: int64

## Network islands
The operational NH graph (junction to junction, links included). Islands are either real gaps or places where OSM tagging breaks the chain.

In [5]:
isl = pd.read_csv(A / "network_islands.csv")
total = pd.read_parquet(nh.OUT / "nh_graph_edges.parquet").km.sum()
print(f"{SUMMARY['n_components']:,} connected pieces; the largest holds {isl.km.iloc[0] / total:.1%} of graph km")
print(f"{SUMMARY['graph_gap_edges']:,} gap edges join dead ends lying within {nh.CFG['graph_snap_m']} m of another NH node")
isl.head(10)

683 connected pieces; the largest holds 98.6% of graph km
7,403 gap edges join dead ends lying within 50 m of another NH node


,component,nodes,km,lon,lat
0,0,209390,"201,658.02",78.48,21.76
1,1,342,535.25,82.55,17.83
2,2,113,60.94,77.03,8.39
3,3,95,27.81,76.90,8.52
4,4,79,41.49,72.53,23.10
5,5,65,30.97,77.46,16.25
6,6,61,223.81,92.88,12.64
7,7,59,21.07,77.88,12.91
8,8,45,46.85,76.78,8.75
9,9,40,100.77,92.70,11.71
